In [ ]:
# ideia:
# arvore de decisão pra qual N chutar. Pra cada N chutado e cada uma das 3 outcomes, atualizamos um vetor da probabilidade de cada p (que começa como [0.1] * 10), e calculamos a diff de entropia.
# escolhemos o n que maximiza o ganho de informacao (perda de entropia)

In [11]:
import numpy as np
import bisect
import math

MAX_N = 100_000
P_VALUES = list(range(1, 11))

# calcular a ground truth
true_sets = []
for p in P_VALUES:
    if p == 1:
        true_sets.append(list(range(1, MAX_N + 1))) # p=1 é true pra td
    else:
        max_base = int(round(MAX_N ** (1.0 / p))) + 2
        lst = [m ** p for m in range(1, max_base) if m ** p <= MAX_N]
        true_sets.append(lst)

outcome_table = np.zeros((MAX_N + 1, 10), dtype=np.uint8)

def nearest_info(x, lst):
    pos = bisect.bisect_left(lst, x) # mesma tatica q usa no game.py
    has_larger = pos < len(lst)
    dist_larger = lst[pos] - x if has_larger else float('inf')
    has_smaller = pos > 0
    dist_smaller = x - lst[pos - 1] if has_smaller else float('inf')
    return has_smaller, dist_smaller, has_larger, dist_larger

# pre-calcular as outcomes pra cada guess de n pra ir mais rapido
for n in range(1, MAX_N + 1):
    for idx, lst in enumerate(true_sets):

        if len(lst) == MAX_N:
            outcome_table[n, idx] = 0
            continue

        pos = bisect.bisect_left(lst, n)
        if pos < len(lst) and lst[pos] == n:
            outcome_table[n, idx] = 0 # igual
            continue

        has_smaller, d_small, has_larger, d_large = nearest_info(n, lst)
        if not has_smaller:
            outcome_table[n, idx] = 2 # maior
        elif not has_larger:
            outcome_table[n, idx] = 1 # menor
        else:
            if d_small < d_large:
                outcome_table[n, idx] = 1 # menor
            elif d_large < d_small:
                outcome_table[n, idx] = 2 # maior
            else:
                outcome_table[n, idx] = 1 # empate -> menor (de acordo com ojogo)

class GameState:
    def __init__(self, prob=None):
        if prob is None:
            self.prob = np.full(10, 1.0 / 10)
        else:
            self.prob = prob

    def system_entropy(self):
        p = self.prob[self.prob > 0]
        return -np.sum(p * np.log2(p)) if len(p) > 0 else 0.0

    def update(self, n, outcome_char):
        out = outcome_table[n]
        if outcome_char == 'igual':
            self.prob *= (out == 0)
        elif outcome_char == 'menor':
            self.prob *= (out == 1)
        else:
            self.prob *= (out == 2)
        s = self.prob.sum()
        if s > 0:
            self.prob /= s
        # nao sei se esse calculo é mto naive mas funcionoubem

def get_outcome_entropy(prob, active_indices):
    p_T = np.zeros(MAX_N + 1)
    p_S = np.zeros(MAX_N + 1)
    p_L = np.zeros(MAX_N + 1)
    for i in active_indices:
        p_val = prob[i]
        col = outcome_table[:, i]
        p_T += (col == 0) * p_val
        p_S += (col == 1) * p_val
        p_L += (col == 2) * p_val
    H = np.zeros(MAX_N + 1)
    for p_dist in (p_T, p_S, p_L):
        mask = p_dist > 1e-12
        H[mask] -= p_dist[mask] * np.log2(p_dist[mask])
    return H

def build_tree(prob, taken=None, depth=0):
    if taken is None:
        taken = set()

    # primeiro vamos ver quais indices ainda tem chance de ser (treshold é 1e-12 pra evitar erros de precisao)
    active_indices = np.where(prob > 1e-12)[0]
    num_active = len(active_indices)
    indent = "  " * depth

    # chegamos numa leaf pq só tem uma chance não-zero (entropia=0)
    if num_active <= 1:
        idx = active_indices[0] if num_active == 1 else np.argmax(prob)
        p_val = P_VALUES[idx]
        print(f"{indent[:2]}|-- [leaf] perfect {p_val}-th power")
        return {'type': 'rule', 'p': p_val}

    # se nao, calculamos a entorpia atual do sistema e crimaos um vetor das entropias de cada guess pra cada outcome (igual, menor, maior, pra todos os ns)
    current_entropy = -np.sum(prob[active_indices] * np.log2(prob[active_indices]))
    H_O = get_outcome_entropy(prob, active_indices)
    H_O[0] = -1.0
    if taken:
        H_O[list(taken)] = -1.0 # p nao ser pego no argmax do melhor n
    best_n = int(np.argmax(H_O))
    expected_gain = max(0, current_entropy - H_O[best_n])
    print(f"{indent}|--- [node] depth {depth} | n={best_n} | entropy: {current_entropy:.4f} | exp. gain: {expected_gain:.4f}")
    node = {'type': 'number', 'n': best_n, 'branches': {}}

    # agora, criamos mais um node pra cada uma das branches possíveis recursivamente
    # entao pra cada opcao (igual, menor, maior), vamos pegar uma mask, atualizar as probs, somar tudo e normalizar, e passar pra recursao
    for char, code in [('igual', 0), ('menor', 1), ('maior', 2)]:
        mask = outcome_table[best_n, :] == code
        new_prob = prob * mask
        total = new_prob.sum()
        if total > 1e-12:
            print(f"{indent}│   ({char})")
            node['branches'][char] = build_tree(new_prob / total, taken | {best_n}, depth + 1)
    # agr o node teoricamente ja tem as suas branches construidas recursivamente, ent volta pro caller
    return node


initial_prob = np.full(10, 1.0 / 10) # inicialmente uniforme
tree = build_tree(initial_prob)

|--- [node] depth 0 | n=512 | entropy: 3.3219 | exp. gain: 1.7510
│   (igual)
  |--- [node] depth 1 | n=5 | entropy: 1.5850 | exp. gain: 0.0000
  │   (igual)
  |-- [leaf] perfect 1-th power
  │   (menor)
  |-- [leaf] perfect 9-th power
  │   (maior)
  |-- [leaf] perfect 3-th power
│   (menor)
  |--- [node] depth 1 | n=243 | entropy: 2.0000 | exp. gain: 0.5000
  │   (igual)
  |-- [leaf] perfect 5-th power
  │   (menor)
    |--- [node] depth 2 | n=65 | entropy: 1.0000 | exp. gain: 0.0000
    │   (menor)
  |-- [leaf] perfect 10-th power
    │   (maior)
  |-- [leaf] perfect 7-th power
  │   (maior)
  |-- [leaf] perfect 8-th power
│   (maior)
  |--- [node] depth 1 | n=9 | entropy: 1.5850 | exp. gain: 0.0000
  │   (igual)
  |-- [leaf] perfect 2-th power
  │   (menor)
  |-- [leaf] perfect 6-th power
  │   (maior)
  |-- [leaf] perfect 4-th power


In [12]:
init_entropy = GameState().system_entropy()
g = init_entropy / np.log2(3) # 3 possibilidades de resposta (igual, maior, menor)
g

np.float64(2.095903274289385)

In [ ]:
# como g é 2.09, o numero mínimo de guesses pra determinar o sistema é ceil(2.09) = 3. Entao a tree q construímos é ótima
# na vdd pode ser que tenha uma tree mais ótima se a profundidade média dela for menor.
# mas aí o avg depth teria que ser g pra ser ótimo ne? vo calcula ele

In [14]:
def get_leaf_depths(node, current_depth=0):
    depths = []
    if node['type'] == 'rule':
        return [current_depth]

    for branch in node['branches'].values():
        depths.extend(get_leaf_depths(branch, current_depth + 1))
    return depths

depths = get_leaf_depths(tree)
avg_depth = sum(depths) / len(depths)

print(f"{avg_depth:.2f}")

2.20


In [ ]:
# bom o suficiente
# mas cpa q se nao usar metodo greedy funcione, e como buildo rapido da pra otimizar assim
# como o split divide em 3, o ganho maximo de entropia em cada step seria log2(3) bits = 1.58, o q splitaria o dset em 1/3, 1/3 e 1/3
# ai teria 9 leaves com depth 2 e uma com depth 3
# mas ai o depth seria 2.1 nao 2.09 oq é estranho, sla deve tar certo
# mas entao precisamos de um split que de 1.58 de entropia em cada um. mas o nosso algo ja faz isso (nao exatamente 1.58, mas ele pega o melhor)
# ue
# ah precisamos minimizar a entropia no sistema, nao na guess, então tudo bem a gente pegar uma leaf que dá menos info no começo se ela compensar dps
# mas da pra fazer brute force (construir todas as arvores e ver o depth médio de cada uma)? hmm
# pior q isso so daria ganho de 0.03 na media final no mlr dos casos


In [23]:
import numpy as np

from tqdm.notebook import tqdm

memo = {}

def get_best_tree(active_indices):
    if len(active_indices) <= 1:
        idx = active_indices[0]
        return 0, {'type': 'rule', 'p': P_VALUES[idx]}

    state = tuple(sorted(active_indices))
    if state in memo:
        return memo[state]

    best_avg_cost = float('inf')
    best_node = None

    for n in range(1, 100_000 + 1):
        branches = {}
        expected_remaining_depth = 0

        valid_n = False
        for char, code in [('igual', 0), ('menor', 1), ('maior', 2)]:
            branch_indices = [i for i in active_indices if outcome_table[n, i] == code]

            if len(branch_indices) == 0:
                continue
            if len(branch_indices) == len(active_indices):
                break # n inutil

            valid_n = True
            cost, subtree = get_best_tree(branch_indices)

            prob = len(branch_indices) / len(active_indices)
            expected_remaining_depth += prob * cost
            branches[char] = subtree
        else:
            if not valid_n: continue

            total_cost = 1 + expected_remaining_depth

            if total_cost < best_avg_cost:
                best_avg_cost = total_cost
                best_node = {'type': 'number', 'n': n, 'branches': branches}

    memo[state] = (best_avg_cost, best_node)
    return best_avg_cost, best_node

initial_indices = list(range(10))
opt_avg_depth, optimal_tree = get_best_tree(initial_indices)

print(f"{opt_avg_depth:.4f}")

2.2000


In [ ]:
# como o codigo acima retornou 2.2, teoricamente essa profundidade é a matematicamente ótima. entao nossa tree ta otima

In [15]:
np.log2(3)

np.float64(1.584962500721156)

In [10]:

str(tree)

"{'type': 'number', 'n': 512, 'branches': {'igual': {'type': 'number', 'n': 5, 'branches': {'igual': {'type': 'rule', 'p': 1}, 'menor': {'type': 'rule', 'p': 9}, 'maior': {'type': 'rule', 'p': 3}}}, 'menor': {'type': 'number', 'n': 243, 'branches': {'igual': {'type': 'rule', 'p': 5}, 'menor': {'type': 'number', 'n': 65, 'branches': {'menor': {'type': 'rule', 'p': 10}, 'maior': {'type': 'rule', 'p': 7}}}, 'maior': {'type': 'rule', 'p': 8}}}, 'maior': {'type': 'number', 'n': 9, 'branches': {'igual': {'type': 'rule', 'p': 2}, 'menor': {'type': 'rule', 'p': 6}, 'maior': {'type': 'rule', 'p': 4}}}}}"